In [1]:
import os, warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
print("LANGSMITH_TRACING :", os.getenv("LANGSMITH_TRACING", "not set"))
print("LANGSMITH_API_KEY :", "✅" if os.getenv("LANGSMITH_API_KEY") else "❌  missing")
print("LANGSMITH_PROJECT :", os.getenv("LANGSMITH_PROJECT", "default"))
print("GROQ_API_KEY      :", "✅" if os.getenv("GROQ_API_KEY")      else "❌  missing")
print("GOOGLE_API_KEY    :", "✅" if os.getenv("GOOGLE_API_KEY")    else "❌  missing")

LANGSMITH_TRACING : true
LANGSMITH_API_KEY : ✅
LANGSMITH_PROJECT : Langsmith_Practise_Krish_Naik
GROQ_API_KEY      : ✅
GOOGLE_API_KEY    : ✅


In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

# Plain llm.invoke() — no LCEL, no chain, no decorators.
# LangSmith intercepts this call automatically via the three env vars we set.
response = llm.invoke("What is a LangSmith Run? Answer in 2 sentences.")
print(response.content)

A LangSmith Run is a recorded execution of a LangChain application that captures all prompts, responses, and metadata for each step of the workflow. It enables developers to monitor, debug, and analyze the performance and behavior of their language‑model pipelines in a centralized dashboard.


## Custom

In [11]:
import re
from langsmith import traceable


# ── Tool: keyword search over the real document ────────────────────────────
@traceable(run_type="tool", name="doc_keyword_search")
def search_document(query: str, top_k: int = 3) -> list:
    """Searches llm_production_guide.txt by keyword overlap. Visible as a Tool Run."""
    with open("../data/llm_production_guide.txt", encoding="utf-8") as f:
        text = f.read()
    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 80]
    keywords   = set(re.findall(r"\b\w{4,}\b", query.lower()))
    ranked     = sorted(paragraphs,
                        key=lambda p: sum(1 for kw in keywords if kw in p.lower()),
                        reverse=True)
    return ranked[:top_k]


In [12]:
# ── Chain: orchestrates search → LLM → answer ─────────────────────────────
@traceable(run_type="chain", name="doc_qa_pipeline")
def doc_qa(question: str) -> str:
    """Parent chain. LangSmith shows: doc_qa_pipeline → doc_keyword_search + ChatGroq."""
    sections = search_document(question)            # ← child Tool Run
    context  = "\n\n".join(sections)
    prompt   = f"Context:\n{context}\n\nQuestion: {question}\nAnswer concisely:"
    return llm.invoke(prompt).content               # ← child LLM Run

In [13]:
answer = doc_qa("What are the main LLM security threats?")
print(f"Answer: {answer[:300]}...")

Answer: **Main LLM security threats (as highlighted by the OWASP Top 10 for LLM Applications and production‑security guides)**  

1. **Prompt / Injection Attacks** – Manipulating prompts to cause the model to reveal secrets, execute unintended actions, or produce harmful output.  
2. **Data Leakage / Privac...


## Production Level MetaData and Tags

In [14]:
from langsmith import get_current_run_tree 

@traceable(run_type="chain", name="support-query")
def support_qa(question: str, user_id: str, session_id: str) -> str:
    run = get_current_run_tree()
    if run:
        run.metadata.update({
            "user_id":    user_id,
            "session_id": session_id,
            "feature":    "customer-support",
            "env":        "production",
        })
        run.tags = ["production", "support-bot", "groq"]
    return llm.invoke(question).content

In [15]:
list_of_queriers = [
    ("priya",   "sess_001", "What is prompt injection and how do we prevent it?"),
    ("aditi",   "sess_002", "What are best practices for LLM output validation?"),
    ("sheetal", "sess_003", "How do we monitor LLM costs in production?"),
]

In [16]:
# Run three different users — each trace is tagged for filtering
for user, session, q in list_of_queriers:
    answer = support_qa(q, user_id=user, session_id=session)
    print(f"[{user}] {answer[:150]}...\n")

[priya] ## Prompt Injection – A Quick Primer

**Prompt injection** is a class of attacks (or accidental mishaps) that manipulate the text that an LLM receives...

[aditi] ## Overview  

Validating the output of a large language model (LLM) is essential whenever the model’s responses are used for decision‑making, user‑fa...

[sheetal] ## Monitoring LLM Costs in Production – A Play‑by‑Play Guide  

Below is a **complete, end‑to‑end framework** you can adopt (or cherry‑pick) to keep y...

